# SoftMeta Chatterbox TTS Server v0.2.0

This notebook uses only the SoftMeta repositories and the official Resemble AI
Chatterbox package. Select an **L4 GPU** before running every cell from top to bottom.

- Engine: `soft-meta/chatterbox-v2@v0.2.0`
- Server and UI: `soft-meta/Chatterbox-TTS-Server@v0.2.0`

In [ ]:
%%bash
set -euo pipefail

apt-get update -qq
apt-get install -y -qq ffmpeg libsndfile1 git curl ca-certificates lsof

cd /content
rm -rf /content/bin
mkdir -p /content/bin

MICROMAMBA="/content/bin/micromamba"
MICROMAMBA_VERSION="2.6.2-1"
MICROMAMBA_URL="https://github.com/mamba-org/micromamba-releases/releases/download/${MICROMAMBA_VERSION}/micromamba-linux-64"

echo "Downloading micromamba ${MICROMAMBA_VERSION}..."
curl --fail --location --retry 5 --retry-delay 2 --retry-all-errors \
  --connect-timeout 30 "${MICROMAMBA_URL}" --output "${MICROMAMBA}"
chmod +x "${MICROMAMBA}"
"${MICROMAMBA}" --version

if "${MICROMAMBA}" env list | grep -q 'sm311'; then
  "${MICROMAMBA}" env remove -n sm311 -y || true
fi

"${MICROMAMBA}" create -y -n sm311 -c conda-forge python=3.11 pip
echo "Python 3.11 environment is ready."

In [ ]:
%%bash
set -euo pipefail

MM="/content/bin/micromamba"
cd /content
rm -rf chatterbox-v2 Chatterbox-TTS-Server

git clone --branch v0.2.0 --depth 1 https://github.com/soft-meta/chatterbox-v2.git
git clone --branch v0.2.0 --depth 1 https://github.com/soft-meta/Chatterbox-TTS-Server.git

"$MM" run -n sm311 python -m pip install -U pip setuptools wheel

echo "Installing CUDA PyTorch for Colab L4..."
"$MM" run -n sm311 python -m pip install \
  --index-url https://download.pytorch.org/whl/cu124 \
  torch==2.6.0 torchaudio==2.6.0

echo "Installing the official Chatterbox package..."
"$MM" run -n sm311 python -m pip install --no-cache-dir chatterbox-tts==0.1.7

echo "Installing the SoftMeta engine adapter without changing CUDA PyTorch..."
"$MM" run -n sm311 python -m pip install --no-deps -e /content/chatterbox-v2

echo "Installing the SoftMeta server..."
"$MM" run -n sm311 python -m pip install -r /content/Chatterbox-TTS-Server/requirements-colab.txt

echo "Installation completed."

In [ ]:
%%bash
set -euo pipefail
/content/bin/micromamba run -n sm311 python - <<'PYVERIFY'
import sys
import torch
import torchaudio
import chatterbox
from softmeta_chatterbox import SoftMetaChatterboxEngine

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("CUDA is unavailable. Change the Colab runtime to an L4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("Official Chatterbox package:", chatterbox.__file__)
runtime = SoftMetaChatterboxEngine(device="auto")
print("SoftMeta engine device:", runtime.device)
print("Environment verification passed.")
PYVERIFY

In [ ]:
import os
import signal
import socket
import subprocess
import time
from pathlib import Path
from IPython.display import HTML, display

PORT = 8004
PROJECT = Path('/content/Chatterbox-TTS-Server')
LOG = Path('/content/softmeta_chatterbox_v020.log')
PID_FILE = Path('/content/softmeta_chatterbox_v020.pid')
MM = '/content/bin/micromamba'

if PID_FILE.exists():
    try:
        os.kill(int(PID_FILE.read_text().strip()), signal.SIGTERM)
        time.sleep(1)
    except Exception:
        pass
subprocess.run(f"lsof -t -i:{PORT} | xargs -r kill -9", shell=True, check=False)
LOG.unlink(missing_ok=True)

env = {
    **os.environ,
    'PYTHONUNBUFFERED': '1',
    'HF_HOME': '/content/hf_home',
    'HF_HUB_CACHE': '/content/hf_home/hub',
    'TRANSFORMERS_CACHE': '/content/hf_home/transformers',
    'SOFTMETA_DEVICE': 'cuda',
    'SOFTMETA_MODEL': 'chatterbox',
}
Path(env['HF_HOME']).mkdir(parents=True, exist_ok=True)

log_handle = LOG.open('w', encoding='utf-8', errors='replace')
process = subprocess.Popen(
    [MM, 'run', '-n', 'sm311', 'python', '-u', 'start.py'],
    cwd=PROJECT,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
PID_FILE.write_text(str(process.pid), encoding='utf-8')

def port_open() -> bool:
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=.5):
            return True
    except OSError:
        return False

print('Starting SoftMeta Chatterbox TTS Server...')
for _ in range(300):
    if process.poll() is not None:
        log_handle.flush()
        raise RuntimeError(LOG.read_text(errors='replace')[-16000:])
    if port_open():
        break
    time.sleep(1)
else:
    raise TimeoutError('The server did not open port 8004. Run the log cell below.')

from google.colab.output import eval_js
url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
html = f'''<p><a href="{url}" target="_blank" style="display:inline-block;padding:13px 19px;background:#5f52e8;color:#fff;border-radius:8px;text-decoration:none;font-weight:700">Open SoftMeta Chatterbox TTS Server</a></p><p style="font-size:13px;color:#667085">The Original model loads in the background. The first download may take several minutes.</p>'''
display(HTML(html))
print('Server PID:', process.pid)
print('Server log:', LOG)

## Recent server log

In [ ]:
from pathlib import Path
log = Path('/content/softmeta_chatterbox_v020.log')
print(log.read_text(errors='replace')[-20000:] if log.exists() else 'No server log yet.')

## Stop the server

In [ ]:
import os
import signal
import subprocess
from pathlib import Path
pid_file = Path('/content/softmeta_chatterbox_v020.pid')
if pid_file.exists():
    try:
        os.kill(int(pid_file.read_text().strip()), signal.SIGTERM)
    except Exception:
        pass
    pid_file.unlink(missing_ok=True)
subprocess.run('lsof -t -i:8004 | xargs -r kill -9', shell=True, check=False)
print('Server stopped.')